In [ ]:
%pip install -q transformers==4.57.1 trl==0.23.1 peft==0.17.1 accelerate==1.10.1 bitsandbytes==0.46.1 datasets==4.0.0 scikit-learn==1.7.2


In [ ]:
import os
os.environ['PYTHONHASHSEED'] = '42'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import json
import pickle
import random
import zipfile
from pathlib import Path
import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from scipy.sparse import hstack
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
assert torch.cuda.is_available()
print({'seed': SEED, 'gpu': torch.cuda.get_device_name(0), 'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)})


In [ ]:
work_root = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
data_root = work_root / 'aa_input'
data_root.mkdir(parents=True, exist_ok=True)
roots = [Path('/kaggle/input'), Path('/content'), Path.cwd()]
required = ['kid_adult.jsonl', 'public_test_style.jsonl', 'style_clf.pkl', 'good_bad.jsonl', 'public_test_quality.jsonl']
located = {name: sorted({path for root in roots if root.exists() for path in root.rglob(name)}) for name in required}
if all(len(paths) == 1 for paths in located.values()):
    for name, paths in located.items():
        (data_root / name).write_bytes(paths[0].read_bytes())
else:
    archive_name = 'ml-olympiad-red-task-c1005bf0-8695-451a-9616-87c8687dce27.zip'
    archives = sorted({path for root in roots if root.exists() for path in root.rglob(archive_name)})
    if len(archives) != 1:
        raise FileNotFoundError({'files': {name: [str(path) for path in paths] for name, paths in located.items()}, 'archives': [str(path) for path in archives]})
    members = {
        'ml-olympiad-red-task/data/kid_adult.jsonl': 'kid_adult.jsonl',
        'ml-olympiad-red-task/data/public_test_style.jsonl': 'public_test_style.jsonl',
        'ml-olympiad-red-task/metrics/style_clf.pkl': 'style_clf.pkl',
        'ml-olympiad-red-task/data/good_bad.jsonl': 'good_bad.jsonl',
        'ml-olympiad-red-task/data/public_test_quality.jsonl': 'public_test_quality.jsonl'
    }
    with zipfile.ZipFile(archives[0]) as source:
        for member, destination in members.items():
            (data_root / destination).write_bytes(source.read(member))
print({'files': sorted(path.name for path in data_root.iterdir())})


In [ ]:
def read_jsonl(path):
    with path.open(encoding='utf-8') as source:
        return [json.loads(line) for line in source]
train_rows = read_jsonl(data_root / 'kid_adult.jsonl')
test_rows = read_jsonl(data_root / 'public_test_style.jsonl')
train_dataset = Dataset.from_list([
    {
        'prompt': [{'role': 'user', 'content': row['prompt']}],
        'completion': [{'role': 'assistant', 'content': row['kid']}]
    }
    for row in train_rows
])
print({'train_rows': len(train_dataset), 'test_rows': len(test_rows)})


In [ ]:
model_id = 'Qwen/Qwen3-4B-Instruct-2507'
compute_dtype = torch.float16
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization,
    torch_dtype=compute_dtype,
    device_map='auto'
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(
    task_type='CAUSAL_LM',
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


In [ ]:
run_root = work_root / 'aa_sft'
training_args = SFTConfig(
    output_dir=str(run_root),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy='no',
    report_to='none',
    optim='paged_adamw_8bit',
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    max_length=384,
    packing=False,
    completion_only_loss=True,
    seed=SEED,
    data_seed=SEED
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer
)
result = trainer.train()
adapter_root = work_root / 'artifacts' / 'sft_adapter'
trainer.model.save_pretrained(adapter_root)
tokenizer.save_pretrained(adapter_root)
print({'train_loss': round(result.training_loss, 6), 'adapter': str(adapter_root)})


In [ ]:
def generate_style(active_model):
    active_model.config.use_cache = True
    active_model.eval()
    generated_texts = []
    for row in test_rows:
        inputs = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': row['prompt']}],
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors='pt'
        ).to(active_model.device)
        with torch.inference_mode():
            generated = active_model.generate(
                **inputs,
                do_sample=False,
                num_beams=1,
                max_new_tokens=192,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        generated_texts.append(tokenizer.decode(generated[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip())
    return generated_texts
model = trainer.model
outputs = generate_style(model)
assert len(outputs) == len(test_rows)
print({'generation_count': len(outputs), 'do_sample': False})


In [ ]:
with (data_root / 'style_clf.pkl').open('rb') as source:
    style_metric = pickle.load(source)
simple_index = style_metric['clf'].classes_.tolist().index(1)
def score_style(texts):
    features = hstack([vectorizer.transform(texts) for vectorizer in style_metric['vecs']])
    return float(style_metric['clf'].predict_proba(features)[:, simple_index].mean())
def style_interval(value):
    return 'А' if value < 0.4 else 'Б' if value < 0.7 else 'В' if value < 0.9 else 'Г'
p_simple = score_style(outputs)
answer = style_interval(p_simple)
output_root = work_root / 'artifacts'
output_root.mkdir(parents=True, exist_ok=True)
(output_root / 'sft_style_outputs.json').write_text(json.dumps(outputs, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'P_simple={p_simple:.6f}')
print(f'answer={answer}')


In [ ]:
dpo_dataset = Dataset.from_list([
    {
        'prompt': [{'role': 'user', 'content': row['prompt']}],
        'chosen': [{'role': 'assistant', 'content': row['kid']}],
        'rejected': [{'role': 'assistant', 'content': row['adult']}]
    }
    for row in train_rows
])
model.config.use_cache = False
torch.cuda.empty_cache()
reference_base = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization,
    torch_dtype=compute_dtype,
    device_map='auto'
)
reference_model = PeftModel.from_pretrained(reference_base, adapter_root, is_trainable=False)
reference_model.config.use_cache = False
reference_model.requires_grad_(False)
reference_model.eval()
print({'dpo_pairs': len(dpo_dataset), 'reference': str(adapter_root)})


In [ ]:
dpo_run_root = work_root / 'aa_dpo_style'
dpo_args = DPOConfig(
    output_dir=str(dpo_run_root),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy='no',
    report_to='none',
    optim='paged_adamw_8bit',
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    max_length=384,
    max_prompt_length=128,
    max_completion_length=256,
    beta=0.1,
    loss_type='sigmoid',
    seed=SEED,
    data_seed=SEED
)
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=reference_model,
    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer
)
dpo_result = dpo_trainer.train()
dpo_adapter_root = work_root / 'artifacts' / 'dpo_style_adapter'
dpo_trainer.model.save_pretrained(dpo_adapter_root)
tokenizer.save_pretrained(dpo_adapter_root)
print({'dpo_train_loss': round(dpo_result.training_loss, 6), 'adapter': str(dpo_adapter_root)})


In [ ]:
dpo_model = dpo_trainer.model
dpo_trainer.ref_model = None
del reference_model
torch.cuda.empty_cache()
dpo_outputs = generate_style(dpo_model)
dpo_p_simple = score_style(dpo_outputs)
dpo_answer = style_interval(dpo_p_simple)
(output_root / 'dpo_style_outputs.json').write_text(json.dumps(dpo_outputs, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'P_simple_dpo={dpo_p_simple:.6f}')
print(f'answer_dpo={dpo_answer}')


In [ ]:
from torch.utils.data import DataLoader, Dataset as TorchDataset
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch.nn.functional as F
quality_train_rows = read_jsonl(data_root / 'good_bad.jsonl')
quality_test_rows = read_jsonl(data_root / 'public_test_quality.jsonl')
def quality_prompt(row):
    return row.get('instruction', row.get('prompt'))
def reward_tokens(prompt, response, max_length):
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}, {'role': 'assistant', 'content': response}],
        add_generation_prompt=False,
        tokenize=True,
        truncation=True,
        max_length=max_length
    )
class RewardPairDataset(TorchDataset):
    def __init__(self, rows, max_length):
        self.items = [
            {
                'chosen_input_ids': reward_tokens(quality_prompt(row), row['chosen'], max_length),
                'rejected_input_ids': reward_tokens(quality_prompt(row), row['rejected'], max_length)
            }
            for row in rows
        ]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, index):
        return self.items[index]
class RewardPairCollator:
    def __call__(self, features):
        chosen = tokenizer.pad({'input_ids': [item['chosen_input_ids'] for item in features]}, padding=True, return_tensors='pt')
        rejected = tokenizer.pad({'input_ids': [item['rejected_input_ids'] for item in features]}, padding=True, return_tensors='pt')
        return {
            'chosen_input_ids': chosen['input_ids'],
            'chosen_attention_mask': chosen['attention_mask'],
            'rejected_input_ids': rejected['input_ids'],
            'rejected_attention_mask': rejected['attention_mask']
        }
class PairwiseRewardTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        chosen_scores = model(
            input_ids=inputs['chosen_input_ids'],
            attention_mask=inputs['chosen_attention_mask']
        ).logits.view(-1)
        rejected_scores = model(
            input_ids=inputs['rejected_input_ids'],
            attention_mask=inputs['rejected_attention_mask']
        ).logits.view(-1)
        loss = -F.logsigmoid(chosen_scores - rejected_scores).mean()
        if return_outputs:
            return loss, {'chosen_scores': chosen_scores, 'rejected_scores': rejected_scores}
        return loss
reward_max_length = 384
reward_train_dataset = RewardPairDataset(quality_train_rows, reward_max_length)
reward_test_dataset = RewardPairDataset(quality_test_rows, reward_max_length)
reward_collator = RewardPairCollator()
print({'reward_train_pairs': len(reward_train_dataset), 'reward_test_pairs': len(reward_test_dataset), 'max_length': reward_max_length})


In [ ]:
del dpo_trainer, dpo_model, model, trainer, reference_base
torch.cuda.empty_cache()
reward_model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=1,
    quantization_config=quantization,
    torch_dtype=compute_dtype,
    device_map='auto'
)
reward_model.config.pad_token_id = tokenizer.pad_token_id
reward_model.config.use_cache = False
reward_model = prepare_model_for_kbit_training(reward_model)
reward_lora = LoraConfig(
    task_type='SEQ_CLS',
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    modules_to_save=['score']
)
reward_model = get_peft_model(reward_model, reward_lora)
reward_model.print_trainable_parameters()
reward_args = TrainingArguments(
    output_dir=str(work_root / 'aa_reward_model'),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy='no',
    report_to='none',
    optim='paged_adamw_8bit',
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    seed=SEED,
    data_seed=SEED
)
reward_trainer = PairwiseRewardTrainer(
    model=reward_model,
    args=reward_args,
    train_dataset=reward_train_dataset,
    data_collator=reward_collator
)
reward_result = reward_trainer.train()
reward_adapter_root = output_root / 'reward_adapter'
reward_trainer.model.save_pretrained(reward_adapter_root)
tokenizer.save_pretrained(reward_adapter_root)
print({'reward_train_loss': round(reward_result.training_loss, 6), 'adapter': str(reward_adapter_root)})


In [ ]:
def score_reward_pairs(active_model, dataset):
    active_model.eval()
    device = next(active_model.parameters()).device
    loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=reward_collator)
    comparisons = []
    for batch in loader:
        with torch.inference_mode():
            chosen_scores = active_model(
                input_ids=batch['chosen_input_ids'].to(device),
                attention_mask=batch['chosen_attention_mask'].to(device)
            ).logits.view(-1)
            rejected_scores = active_model(
                input_ids=batch['rejected_input_ids'].to(device),
                attention_mask=batch['rejected_attention_mask'].to(device)
            ).logits.view(-1)
        comparisons.extend([
            {'chosen_score': float(chosen), 'rejected_score': float(rejected)}
            for chosen, rejected in zip(chosen_scores.cpu(), rejected_scores.cpu())
        ])
    accuracy = sum(item['chosen_score'] > item['rejected_score'] for item in comparisons) / len(comparisons)
    return float(accuracy), comparisons
def reward_interval(value):
    return 'А' if value < 0.6 else 'Б' if value < 0.75 else 'В' if value < 0.9 else 'Г'
reward_pairwise_accuracy, reward_comparisons = score_reward_pairs(reward_trainer.model, reward_test_dataset)
(output_root / 'reward_quality_scores.json').write_text(json.dumps(reward_comparisons, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'pairwise_accuracy={reward_pairwise_accuracy:.6f}')
print(f'answer_rm={reward_interval(reward_pairwise_accuracy)}')
